# P002 指派问题 · PySCIPOpt 建模与评测演示

本 Notebook 运行在 JupyterLite (WebAssembly + PySCIPOpt) 环境中。
示范使用 PySCIPOpt 建立 0-1 整数线性规划模型求解标准指派问题。

In [ ]:
from pyscipopt import Model, quicksum
import numpy as np
print("PySCIPOpt 与 NumPy 加载成功！")

## 1. 契约定义：solve(instance) -> dict

输入数据规范：
- `n`: 人数与任务数
- `costs`: n x n 费用矩阵，`costs[i][j]` 为人 i 完成任务 j 的成本

返回值规范：
- `status`: `"optimal"` 等
- `objective`: float 最小总成本
- `solution`: n x n 矩阵，每行每列恰有一个 1.0，其余为 0.0

In [ ]:
def solve(instance: dict) -> dict:
    n = instance["n"]
    costs = instance["costs"]

    model = Model("assignment")
    model.hideOutput(True)

    # x[i, j] = 1 表示指派人员 i 执行任务 j
    x = {}
    for i in range(n):
        for j in range(n):
            x[i, j] = model.addVar(vtype="B", name=f"x_{i}_{j}")

    # 目标函数：最小化总费用
    model.setObjective(quicksum(costs[i][j] * x[i, j] for i in range(n) for j in range(n)), "minimize")

    # 每个人恰好分配一个任务
    for i in range(n):
        model.addCons(quicksum(x[i, j] for j in range(n)) == 1)

    # 每项任务恰好分配给一个人
    for j in range(n):
        model.addCons(quicksum(x[i, j] for i in range(n)) == 1)

    model.optimize()

    sol = [[float(round(model.getVal(x[i, j]))) for j in range(n)] for i in range(n)]
    return {
        "status": "optimal" if model.getStatus() == "optimal" else str(model.getStatus()),
        "objective": float(model.getObjVal()),
        "solution": sol,
    }

In [ ]:
# 测试用例验证
test_instance = {
    "n": 3,
    "costs": [
        [9, 2, 7],
        [6, 4, 3],
        [5, 8, 1]
    ]
}
res = solve(test_instance)
print("指派求解结果:", res)